In [6]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import random
import time
import json

print("Loading sparse matrix...")
mat = sp.load_npz('/Users/aliakbar/Dev/ebookreader-ncf/user_items.npz')
print(f"Matrix shape: {mat.shape}")
print(f"Non-zeros: {mat.nnz:,}")

# Device — MPS for Apple Silicon
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

Loading sparse matrix...
Matrix shape: (703472, 20000)
Non-zeros: 70,386,166
Device: mps


In [7]:
class NCF(nn.Module):
    def __init__(self, n_users, n_books, embedding_dim=64, layers=[128, 64, 32], dropout=0.2):
        super().__init__()

        # Embedding tables: one learned vector per user, one per book
        # These play the same role as your ALS user_factors and item_factors matrices
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.book_embedding = nn.Embedding(n_books, embedding_dim)

        # Initialize embeddings with small random values (standard practice)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.book_embedding.weight, std=0.01)

        # MLP layers: gradually compress concatenated vectors down to a single score
        # Input size is embedding_dim * 2 because we concatenate user + book vectors
        mlp_layers = []
        input_size = embedding_dim * 2

        for output_size in layers:
            mlp_layers.append(nn.Linear(input_size, output_size))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(dropout))
            input_size = output_size

        # Final layer: compress to a single number, sigmoid squashes it to [0, 1]
        mlp_layers.append(nn.Linear(input_size, 1))
        mlp_layers.append(nn.Sigmoid())

        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, user_ids, book_ids):
        # Look up embedding vectors for this batch of users and books
        user_vec = self.user_embedding(user_ids)    # shape: (batch, embedding_dim)
        book_vec = self.book_embedding(book_ids)    # shape: (batch, embedding_dim)

        # Concatenate: stick the two vectors together end-to-end
        x = torch.cat([user_vec, book_vec], dim=1) # shape: (batch, embedding_dim*2)

        # Pass through MLP, get predicted preference score
        return self.mlp(x).squeeze()               # shape: (batch,)


# Set up device — this will say 'cuda' on Colab with GPU runtime
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Training device: {device}")

Training device: mps


In [8]:
def evaluate(model, val_positives, train_positives, n_books, device, k_values=[5, 10, 20, 50], n_users_eval=2000):
    """
    For each user in the validation set:
    1. Take their 1 held-out book (the correct answer)
    2. Sample 99 random books they haven't interacted with (negatives)
    3. Score all 100 books with the model
    4. Check if the correct book appears in the top K

    This is the standard NCF evaluation protocol from He et al. 2017.
    Note: this is the 100-candidate sampled evaluation (easier than our
    full 20k ALS evaluation), so scores will look higher. We'll note this.
    """
    model.eval()

    hits = {k: 0 for k in k_values}
    mrr_total = 0.0
    n_evaluated = 0

    # Only evaluate on a subset for speed during training
    eval_users = list(val_positives.keys())[:n_users_eval]

    with torch.no_grad():
        for user_idx in eval_users:
            correct_book = val_positives[user_idx]
            pos_set = set(train_positives.get(user_idx, [])) | {correct_book}

            # Sample 99 negative books this user hasn't interacted with
            negs = []
            N_CANDIDATES = 500
            N_NEGATIVES = N_CANDIDATES - 1
            
            while len(negs) < N_NEGATIVES:
                neg = random.randint(0, n_books - 1)
                if neg not in pos_set:
                    negs.append(neg)

            # All 100 candidates: correct book + 99 negatives
            candidates = [correct_book] + negs

            user_tensor = torch.full(
                (len(candidates),),
                user_idx,
                dtype=torch.long,
                device=device
            )
            book_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model(user_tensor, book_tensor).cpu().numpy()

            # Rank candidates by score (highest score = rank 1)
            ranked_indices = scores.argsort()[::-1]  # descending order

            # Find where the correct book (index 0 in candidates) landed
            correct_rank = np.where(ranked_indices == 0)[0][0] + 1  # 1-indexed rank

            # HitRate@K: did the correct book appear in top K?
            for k in k_values:
                if correct_rank <= k:
                    hits[k] += 1

            # MRR: reciprocal of the correct book's rank
            mrr_total += 1.0 / correct_rank
            n_evaluated += 1

    model.train()

    hit_rates = {k: hits[k] / n_evaluated for k in k_values}
    mrr = mrr_total / n_evaluated

    return hit_rates, mrr


n_users = mat.shape[0]
n_books = mat.shape[1]

model = NCF(
    n_users=n_users,
    n_books=n_books,
    embedding_dim=64,
    layers=[128, 64, 32],
    dropout=0.2
).to(device)

# # Quick sanity check: evaluate the untrained model
# # Scores should be near random (~10% Hit@10 with 100 candidates, ~5% MRR)
# print("Evaluating untrained model (expect near-random scores)...")
# hit_rates, mrr = evaluate(model, val_positives, train_positives, n_books, device, n_users_eval=500)
# print(f"Hit@5:  {hit_rates[5]:.4f}  (random baseline: ~0.05)")
# print(f"Hit@10: {hit_rates[10]:.4f}  (random baseline: ~0.10)")
# print(f"Hit@20: {hit_rates[20]:.4f}  (random baseline: ~0.20)")
# print(f"Hit@50: {hit_rates[50]:.4f}  (random baseline: ~0.50)")
# print(f"MRR:    {mrr:.4f}  (random baseline: ~0.05)")

In [9]:
def evaluate_full_catalog(model, val_positives, train_positives,
                          n_books, device, k_values=[5, 10, 20, 50],
                          n_users_eval=1000):
    """
    Full catalog evaluation, comparable to our ALS evaluation.
    For each user: score all 20,000 books, exclude already-read books,
    check if held-out book appears in top K.

    This is the honest evaluation. Numbers will be lower than
    the 100-candidate version — that's correct and expected.
    """
    model.eval()

    hits = {k: 0 for k in k_values}
    mrr_total = 0.0
    n_evaluated = 0

    # Pre-build all book indices once — reused for every user
    all_books = torch.arange(n_books, dtype=torch.long).to(device)

    eval_users = list(val_positives.keys())[:n_users_eval]

    with torch.no_grad():
        for user_idx in eval_users:
            correct_book = val_positives[user_idx]
            already_read = set(train_positives.get(user_idx, []))

            # Score all 20,000 books in one forward pass
            user_tensor = torch.full((n_books,), user_idx,
                                     dtype=torch.long, device=device)
            scores = model(user_tensor, all_books).cpu().numpy()

            # Mask out books the user already interacted with during training
            # Set their score to -infinity so they can't appear in top K
            for read_book in already_read:
                scores[read_book] = -np.inf

            # Rank all books by score
            ranked_indices = scores.argsort()[::-1]

            # Find where the correct held-out book landed
            correct_rank = np.where(ranked_indices == correct_book)[0][0] + 1

            for k in k_values:
                if correct_rank <= k:
                    hits[k] += 1

            mrr_total += 1.0 / correct_rank
            n_evaluated += 1

            if n_evaluated % 100 == 0:
                print(f"  Evaluated {n_evaluated}/{len(eval_users)} users...",
                      flush=True)

    model.train()

    hit_rates = {k: hits[k] / n_evaluated for k in k_values}
    mrr = mrr_total / n_evaluated
    return hit_rates, mrr

In [10]:
import time
import json
import numpy as np
import torch
import torch.nn as nn

def run_experiment(exp_name, n_users_train, embedding_dim, layers,
                   lr, n_epochs, neg_ratio=4):
    """
    Full experiment: load data subset, pre-compute pairs,
    train NCF, evaluate on full catalog, save results.
    """
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {exp_name}")
    print(f"  users={n_users_train}, embedding_dim={embedding_dim}")
    print(f"  layers={layers}, lr={lr}, epochs={n_epochs}")
    print(f"{'='*60}")

    # --- Build user subset ---
    user_counts = np.diff(mat.indptr)
    top_indices = np.sort(np.argsort(user_counts)[-n_users_train:])
    mat_sub = mat[top_indices].tocsr()

    n_users = mat_sub.shape[0]
    n_books = mat_sub.shape[1]

    # --- Train/val split ---
    import random
    train_pos = {}
    val_pos = {}
    for user_idx in range(n_users):
        start = mat_sub.indptr[user_idx]
        end = mat_sub.indptr[user_idx + 1]
        books = mat_sub.indices[start:end].tolist()
        if len(books) < 2:
            continue
        val_book = random.choice(books)
        train_pos[user_idx] = [b for b in books if b != val_book]
        val_pos[user_idx] = val_book

    print(f"Users: {len(train_pos):,}, "
          f"Training positives: {sum(len(v) for v in train_pos.values()):,}")

    # --- Pre-compute training pairs on GPU ---
    print("Pre-computing pairs...")
    user_pos_set = {u: set(b) for u, b in train_pos.items()}
    all_u, all_b, all_l = [], [], []

    for user_idx, pos_books in train_pos.items():
        pos_set = user_pos_set[user_idx]
        for pos_book in pos_books:
            all_u.append(user_idx)
            all_b.append(pos_book)
            all_l.append(1.0)
            negs_added = 0
            while negs_added < neg_ratio:
                neg = random.randint(0, n_books - 1)
                if neg not in pos_set:
                    all_u.append(user_idx)
                    all_b.append(neg)
                    all_l.append(0.0)
                    negs_added += 1

    u_gpu = torch.tensor(all_u, dtype=torch.long).to(device)
    b_gpu = torch.tensor(all_b, dtype=torch.long).to(device)
    l_gpu = torch.tensor(all_l, dtype=torch.float32).to(device)

    ram_gb = (u_gpu.nbytes + b_gpu.nbytes + l_gpu.nbytes) / 1024**3
    print(f"GPU RAM for data: {ram_gb:.2f} GB")
    print(f"GPU RAM total used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    BATCH_SIZE = 4096
    n_batches = (len(u_gpu) + BATCH_SIZE - 1) // BATCH_SIZE

    # --- Build model ---
    model = NCF(n_users, n_books, embedding_dim, layers, dropout=0.2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {total_params:,}")

    # --- Training loop ---
    history = []
    best_mrr = 0
    best_epoch = 0

    for epoch in range(1, n_epochs + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0

        perm = torch.randperm(len(u_gpu), device=device)
        u_s = u_gpu[perm]
        b_s = b_gpu[perm]
        l_s = l_gpu[perm]

        for i in range(0, len(u_s), BATCH_SIZE):
            u = u_s[i:i+BATCH_SIZE]
            b = b_s[i:i+BATCH_SIZE]
            l = l_s[i:i+BATCH_SIZE]
            pred = model(u, b)
            loss = loss_fn(pred, l)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / n_batches
        epoch_time = time.time() - epoch_start

        # Evaluate every 5 epochs and at the end
        if epoch % 5 == 0 or epoch == n_epochs:
            hit_rates, mrr = evaluate(model, val_pos, train_pos,
                                       n_books, device, n_users_eval=1000)
            if mrr > best_mrr:
                best_mrr = mrr
                best_epoch = epoch
                # Save best model weights
                torch.save(model.state_dict(),
                    f'/Users/aliakbar/Dev/ebookreader-ncf/{exp_name}_best.pt')



            history.append({
                'epoch': epoch, 'loss': avg_loss,
                'hit10': hit_rates[10], 'mrr': mrr
            })
            print(f"Epoch {epoch:3d}/{n_epochs} | Loss: {avg_loss:.4f} | "
                  f"Hit@10: {hit_rates[10]:.4f} | MRR: {mrr:.4f} | "
                  f"{epoch_time:.0f}s")
        else:
            print(f"Epoch {epoch:3d}/{n_epochs} | Loss: {avg_loss:.4f} | "
                  f"{epoch_time:.0f}s")

    print(f"\nBest MRR: {best_mrr:.4f} at epoch {best_epoch}")

    # --- Load best model and run full catalog eval ---
    print("\nRunning full 20k catalog evaluation...")
    model.load_state_dict(torch.load(
    f'/Users/aliakbar/Dev/ebookreader-ncf/{exp_name}_best.pt', map_location=device))

    hit_rates_full, mrr_full = evaluate_full_catalog(
        model, val_pos, train_pos, n_books, device, n_users_eval=500)

    # --- Save results ---
    result = {
        'exp_name': exp_name,
        'n_users_train': n_users_train,
        'embedding_dim': embedding_dim,
        'layers': layers,
        'lr': lr,
        'n_epochs': n_epochs,
        'best_sampled_mrr': best_mrr,
        'best_epoch': best_epoch,
        'full_catalog': {
            'hit5': hit_rates_full[5],
            'hit10': hit_rates_full[10],
            'hit20': hit_rates_full[20],
            'hit50': hit_rates_full[50],
            'mrr': mrr_full
        },
        'history': history
    }

    results_path = f'/Users/aliakbar/Dev/ebookreader-ncf/{exp_name}_results.json'
    with open(results_path, 'w') as f:
        json.dump(result, f, indent=2)

    print(f"\n{'='*50}")
    print(f"FULL CATALOG RESULTS — {exp_name}")
    print(f"{'='*50}")
    print(f"Hit@10: {hit_rates_full[10]:.4f}")
    print(f"MRR:    {mrr_full:.4f}  "
          f"(ALS champion: 0.2143)")
    print(f"Results saved to {results_path}")

    # Free GPU memory before next experiment
    del u_gpu, b_gpu, l_gpu, model
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return result


results = []

r = run_experiment(
    exp_name='ncf_20k_256_lr3e3_honest',
    n_users_train=50_000,
    embedding_dim=256,
    layers=[512, 256, 128],
    lr=0.003,
    n_epochs=20,
    neg_ratio=4
)
results.append(r)

print("\n==============================")
print("EXPERIMENT COMPLETE")
print("==============================")


EXPERIMENT: ncf_20k_256_lr3e3_honest
  users=20000, embedding_dim=256
  layers=[512, 256, 128], lr=0.003, epochs=20
Users: 20,000, Training positives: 12,457,413
Pre-computing pairs...
GPU RAM for data: 1.16 GB
GPU RAM total used: 0.00 GB
Model parameters: 10,667,009
Epoch   1/20 | Loss: 0.2983 | 327s
Epoch   2/20 | Loss: 0.2474 | 391s
Epoch   3/20 | Loss: 0.2319 | 399s
Epoch   4/20 | Loss: 0.2222 | 388s
Epoch   5/20 | Loss: 0.2150 | Hit@10: 0.5640 | MRR: 0.3080 | 413s
Epoch   6/20 | Loss: 0.2091 | 413s
Epoch   7/20 | Loss: 0.2041 | 353s
Epoch   8/20 | Loss: 0.1997 | 311s
Epoch   9/20 | Loss: 0.1960 | 313s
Epoch  10/20 | Loss: 0.1925 | Hit@10: 0.6120 | MRR: 0.3335 | 320s
Epoch  11/20 | Loss: 0.1894 | 322s
Epoch  12/20 | Loss: 0.1866 | 329s
Epoch  13/20 | Loss: 0.1841 | 334s
Epoch  14/20 | Loss: 0.1817 | 368s
Epoch  15/20 | Loss: 0.1795 | Hit@10: 0.6090 | MRR: 0.3491 | 387s
Epoch  16/20 | Loss: 0.1775 | 391s
Epoch  17/20 | Loss: 0.1756 | 326s
Epoch  18/20 | Loss: 0.1738 | 316s
Epoch  1

In [12]:
# HONEST EVALUATION — held-out users, remapped to compact indices

print("Building held-out evaluation set...")

user_counts = np.diff(mat.indptr)
top_20k_indices = set(np.argsort(user_counts)[-20_000:].tolist())
mat_csr = mat.tocsr()

# Find 1000 held-out users not in training set, with >= 5 interactions
held_out_raw = []
for user_idx in range(mat.shape[0]):
    if user_idx in top_20k_indices:
        continue
    start = mat_csr.indptr[user_idx]
    end = mat_csr.indptr[user_idx + 1]
    if (end - start) >= 5:
        held_out_raw.append(user_idx)
    if len(held_out_raw) >= 1000:
        break

print(f"Held-out users found: {len(held_out_raw):,}")

# Remap held-out users to indices 20000, 20001, 20002...
# This keeps the embedding table small (21000 rows instead of 703k)
held_out_train = {}
held_out_val = {}

for new_idx, raw_idx in enumerate(held_out_raw):
    remapped_idx = 20_000 + new_idx  # compact new index
    start = mat_csr.indptr[raw_idx]
    end = mat_csr.indptr[raw_idx + 1]
    books = mat_csr.indices[start:end].tolist()
    val_book = random.choice(books)
    held_out_train[remapped_idx] = [b for b in books if b != val_book]
    held_out_val[remapped_idx] = val_book

print(f"Evaluation users: {len(held_out_val):,}")

# Load model — n_users=21000 covers training (0-19999) + held-out (20000-20999)
n_books_full = mat.shape[1]

model_honest = NCF(
    n_users=21_000,
    n_books=n_books_full,
    embedding_dim=256,
    layers=[512, 256, 128],
    dropout=0.2
).to(device)

checkpoint = torch.load(
    '/Users/aliakbar/Dev/ebookreader-ncf/ncf_20k_256_lr3e3_honest_best.pt',
    map_location=device
)

# Save the trained user embeddings
trained_user_emb = checkpoint.pop("user_embedding.weight")

# Load everything else
model_honest.load_state_dict(checkpoint, strict=False)

# Copy the first 20,000 user embeddings into the larger embedding table
with torch.no_grad():
    model_honest.user_embedding.weight[:20000].copy_(trained_user_emb)
model_honest.eval()

print("\nRunning full catalog evaluation on held-out users...")

hit_rates_honest, mrr_honest = evaluate_full_catalog(
    model_honest,
    held_out_val,
    held_out_train,
    n_books_full,
    device,
    n_users_eval=1000
)

print()
print("=" * 55)
print("HONEST NCF RESULTS — unseen users, full 20k catalog")
print("=" * 55)
print(f"Hit@5:  {hit_rates_honest[5]:.4f}")
print(f"Hit@10: {hit_rates_honest[10]:.4f}")
print(f"Hit@20: {hit_rates_honest[20]:.4f}")
print(f"Hit@50: {hit_rates_honest[50]:.4f}")
print(f"MRR:    {mrr_honest:.4f}")
print(f"\nALS champion MRR: 0.2143")

Building held-out evaluation set...
Held-out users found: 1,000
Evaluation users: 1,000

Running full catalog evaluation on held-out users...
  Evaluated 100/1000 users...
  Evaluated 200/1000 users...
  Evaluated 300/1000 users...
  Evaluated 400/1000 users...
  Evaluated 500/1000 users...
  Evaluated 600/1000 users...
  Evaluated 700/1000 users...
  Evaluated 800/1000 users...
  Evaluated 900/1000 users...
  Evaluated 1000/1000 users...

HONEST NCF RESULTS — unseen users, full 20k catalog
Hit@5:  0.0380
Hit@10: 0.0610
Hit@20: 0.0870
Hit@50: 0.1350
MRR:    0.0310

ALS champion MRR: 0.2143
